# Image Quality Model Training & Cascade Pipeline

Trains a binary image-quality classifier (RETFoundGreen backbone) and evaluates it as the first stage of a cascade: images are filtered by predicted quality before being passed to a trained diagnosis model, and cascade performance (coverage, balanced accuracy) is swept across quality-recall thresholds.

In [ ]:
import sys, os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader

from model import UnifiedBackbone
from fundus_dataset import FundusDataset
from img_quality_train_val import train_model, validate, test, find_thresholds_for_recall
from diagnosis_train_eval import validate as validate_diagnosis

## Configuration

In [ ]:
# ── Run configuration ─────────────────────────────────────────────────────
DATASET    = "BRSET"          # "BRSET" or "mBRSET"
MODEL_NAME = "retfound_green"
RUN_ID     = 4

# ── Data paths (edit for your environment) ───────────────────────────────
BRSET_DATA_DIR  = r"C:\Users\preet\Documents\BRSET\data"
MBRSET_DATA_DIR = r"C:\Users\preet\Documents\mBRSET\mBRSET_image_quality\data"
BRSET_IMG_ROOT  = r"C:\Users\preet\Documents\BRSET\data\resized_fundus_photos"
MBRSET_IMG_ROOT = r"C:\Users\preet\Documents\mBRSET\mbrset-a-mobile-brazilian-retinal-dataset-1.0\images"

# Trained diagnosis checkpoint used downstream in the cascade
DIAGNOSIS_CHECKPOINT = {
    "BRSET":  "may24_BRSET__img_diagnosis_model_top1_BA_0.9162.pth",
    "mBRSET": "may24_mBRSET__img_diagnosis_model_top1_BA_0.8202.pth",
}[DATASET]

# Image quality checkpoint is saved/loaded as f"{QUALITY_MODEL_PREFIX}img_quality_model_392.pth"
QUALITY_MODEL_PREFIX = f"IQ_{DATASET}"

RECALL_TARGETS      = [0.99, 0.95, 0.75, 0.55, 0.35, 0.15, 0.05, 0.01]
CASCADE_OUTPUT_DIR  = os.path.join("coverage_graphs", f"cascade_{DATASET.lower()}")

## Load Data

In [ ]:
if DATASET == "BRSET":
    train_df = pd.read_pickle(os.path.join(BRSET_DATA_DIR, "brset_train_524.pkl")).rename(columns={"patient_id": "patient"})
    val_df   = pd.read_pickle(os.path.join(BRSET_DATA_DIR, "brset_val_524.pkl")).rename(columns={"patient_id": "patient"})
    test_df  = pd.read_pickle(os.path.join(BRSET_DATA_DIR, "brset_test_524.pkl")).rename(columns={"patient_id": "patient"})
    img_root = BRSET_IMG_ROOT
else:
    train_df = pd.read_pickle(os.path.join(MBRSET_DATA_DIR, "mbrset_icdr_quality_524_train_full.pkl"))
    val_df   = pd.read_pickle(os.path.join(MBRSET_DATA_DIR, "mbrset_icdr_quality_524_val_full.pkl"))
    test_df  = pd.read_pickle(os.path.join(MBRSET_DATA_DIR, "mbrset_icdr_quality_524_test_full.pkl"))
    img_root = MBRSET_IMG_ROOT

for df in (train_df, val_df, test_df):
    df.dropna(subset=["final_icdr"], inplace=True)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

## Transforms

In [ ]:
if MODEL_NAME != "retfound_green":
    mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
else:
    mean, std = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)

train_tf = A.Compose([
    A.RandomResizedCrop(height=392, width=392, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
    A.HorizontalFlip(),
    A.VerticalFlip(),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(392, 392),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

## Datasets & Loaders

In [ ]:
train_dataset = FundusDataset(train_df, img_root, high_quality_tf=train_tf, low_quality_tf=train_tf, label_col="final_quality")
val_dataset   = FundusDataset(val_df,   img_root, high_quality_tf=val_tf,   low_quality_tf=val_tf,   label_col="final_quality")
test_dataset  = FundusDataset(test_df,  img_root, high_quality_tf=val_tf,   low_quality_tf=val_tf,   label_col="final_quality")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False, num_workers=0)

## Train Image Quality Model

In [ ]:
def compute_class_weights(df, label_col="final_quality"):
    labels  = (df[label_col] > 0).astype(int).values
    counts  = np.bincount(labels)
    weights = 1.0 / counts
    return torch.tensor(weights / weights.sum(), dtype=torch.float32)

device = "cuda"
model  = UnifiedBackbone(model_name=MODEL_NAME)

loss_fn   = nn.CrossEntropyLoss(weight=compute_class_weights(train_df).to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.05)

best_model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=20,
    patience=3,
    str_prefix=QUALITY_MODEL_PREFIX,
)

## Cascade Evaluation

Sweep the quality model's recall thresholds on the (held-out) test set. For each threshold, filter test images by predicted quality and evaluate the trained diagnosis model on the surviving images, recording coverage and diagnostic performance.

In [ ]:
best_model_img_q = UnifiedBackbone(model_name=MODEL_NAME).to(device).float()
best_model_img_q.load_state_dict(torch.load(f"{QUALITY_MODEL_PREFIX}img_quality_model_392.pth"))

best_model_diagnosis = UnifiedBackbone(model_name=MODEL_NAME).to(device).float()
best_model_diagnosis.load_state_dict(torch.load(DIAGNOSIS_CHECKPOINT))

_, val_metrics = validate(best_model_img_q, val_loader, loss_fn, device)
thresholds = find_thresholds_for_recall(val_metrics["all_labels"], val_metrics["all_probs"], RECALL_TARGETS)

# One image per patient for the cascade test set (held-out test split)
cascade_test_df      = test_df.groupby("patient", as_index=False).first()
cascade_test_dataset = FundusDataset(cascade_test_df, img_root, high_quality_tf=val_tf, low_quality_tf=val_tf, label_col="final_quality")
cascade_test_loader  = DataLoader(cascade_test_dataset, batch_size=8, shuffle=False, num_workers=0)

rows = []
for target, thr in thresholds["class_0"].items():
    _, _, good_quality_files = test(best_model_img_q, cascade_test_loader, loss_fn, device, T=thr)

    df_filtered      = cascade_test_df[cascade_test_df["file"].isin(good_quality_files)]
    filtered_dataset = FundusDataset(df_filtered, img_root, high_quality_tf=val_tf, low_quality_tf=val_tf, label_col="final_icdr")
    filtered_loader  = DataLoader(filtered_dataset, batch_size=16, shuffle=False, num_workers=2)

    _, diag_metrics, *_ = validate_diagnosis(best_model_diagnosis, filtered_loader, loss_fn, device)

    rows.append({
        "img_quality_th": thr,
        "recall_low_quality_target": target,
        "coverage": len(df_filtered) / len(cascade_test_df),
        "ba": diag_metrics["ba"],
        "f1": diag_metrics["f1"],
        "n_samp": len(cascade_test_df),
        "n_conf": len(df_filtered),
    })

df_out = pd.DataFrame(rows)
df_out

## Save Results

In [ ]:
os.makedirs(CASCADE_OUTPUT_DIR, exist_ok=True)
out_csv = os.path.join(CASCADE_OUTPUT_DIR, f"cascade_{DATASET}_run{RUN_ID}.csv")
df_out.to_csv(out_csv, index=False)

plt.plot(df_out["coverage"], df_out["ba"], marker="o")
plt.xlabel("Coverage")
plt.ylabel("Balanced Accuracy")
plt.title(f"{DATASET} cascade: BA vs coverage")
plt.show()